# Nanochat Kaggle d8 Run

This notebook is meant for Kaggle.

Attach this Kaggle dataset:

1. `nanochat-codes`

This notebook covers a small-scale end-to-end pipeline:

1. Kaggle setup and dependency install
2. Download a small pretraining dataset slice
3. Train and evaluate the tokenizer
4. Pretrain and evaluate a d8 base model
5. Run supervised fine-tuning and chat evaluation
6. Run distillation
7. Generate preference data and run DPO
8. Compare post-training branches
9. Quantize the distilled model, including a tiny AWQ code-path test
10. Evaluate the quantized model
11. Run tiny RL branch smoke tests
12. Show serving entrypoints for full-precision and quantized models


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CODE_INPUT = Path('/kaggle/input/datasets/yshuaiqin/nanochat-codes')

assert CODE_INPUT.exists(), f'Code dataset not found: {CODE_INPUT}'
print('Code input:', CODE_INPUT)
print('Top-level code files:', sorted(p.name for p in CODE_INPUT.iterdir()))


In [ ]:
WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['NANOCHAT_DTYPE'] = 'float16'
os.environ['PYTHONFAULTHANDLER'] = '1'

print('Working repo:', WORK_REPO)
print('Nanochat base dir:', WORK_CACHE)
print('HF cache:', HF_CACHE)
print('Repo contents:', sorted(p.name for p in WORK_REPO.iterdir()))
print('NANOCHAT_DTYPE:', os.environ['NANOCHAT_DTYPE'])


In [ ]:
# Input HuggingFace token
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

In [ ]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'python-dotenv>=1.2.1',
    'pyyaml>=6.0.0',
    'regex>=2025.9.1',
    'rustbpe>=0.1.0',
    'scipy>=1.15.3',
    'tabulate>=0.9.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)


In [ ]:
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Download a Small Pretraining Slice

Start small for Kaggle. We can scale this up later after the pipeline is stable.


In [ ]:
cmd = [
    sys.executable,
    '-m', 'nanochat.dataset',
    '-n', '8',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 2. Train the Tokenizer


In [ ]:
cmd = [
    sys.executable,
    '-m', 'scripts.tok_train',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 3. Evaluate the Tokenizer


In [ ]:
cmd = [
    sys.executable,
    '-m', 'scripts.tok_eval',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 4. Pretrain a d8 Base Model

This is intentionally conservative for the first Kaggle pass:

- `--depth=8`
- `--nproc_per_node=2` for Kaggle's 2 GPUs
- `--device-batch-size=2` to reduce OOM risk
- short run via `--num-iterations=50`
- disable periodic CORE eval and sampling for now


In [ ]:
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.base_train',
    '--',
    '--depth=8',
    '--model-tag=d8_kaggle',
    '--run=dummy',
    '--device-batch-size=2',
    '--max-seq-len=1024',
    '--num-iterations=50',
    '--eval-every=-1',
    '--core-metric-every=-1',
    '--sample-every=-1',
    '--save-every=-1',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 5. Evaluate the d8 Base Model

Use a reduced 2-GPU evaluation for Kaggle first:

- run with `torchrun --nproc_per_node=2`
- run `core`, `bpb`, and `sample`
- limit CORE examples with `--max-per-task=50`
- reduce BPB tokens with `--split-tokens=262144`
- keep `--device-batch-size=2`


In [ ]:
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.base_eval',
    '--',
    '--eval', 'core,bpb,sample',
    '--model-tag', 'd8_kaggle',
    '--max-per-task', '50',
    '--device-batch-size', '2',
    '--split-tokens', '262144',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 6. Download Identity Conversations for Chat SFT

This small JSONL file gives the model a lightweight assistant identity before chat fine-tuning.


In [ ]:
cmd = [
    'curl',
    '-L',
    '-o', str(WORK_CACHE / 'identity_conversations.jsonl'),
    'https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
print('Saved:', WORK_CACHE / 'identity_conversations.jsonl')


## 7. Chat SFT on Top of the d8 Base Model

Use the Kaggle-safe SFT entrypoint here:

- run `scripts.chat_sft_updated`
- use `torchrun --nproc_per_node=2`
- keep `--run=dummy` to avoid wandb login requirements
- use `--device-batch-size=2`
- set `--total-batch-size=4096` so `--num-iterations=20` behaves as intended
- keep `--mmlu-epochs=1` and `--gsm8k-epochs=1` for a lighter first pass
- disable periodic eval and ChatCORE for this small run


In [ ]:
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_sft_updated',
    '--',
    '--model-tag=d8_kaggle',
    '--run=dummy',
    '--device-batch-size=2',
    '--total-batch-size=4096',
    '--num-iterations=20',
    '--mmlu-epochs=1',
    '--gsm8k-epochs=1',
    '--eval-every=-1',
    '--chatcore-every=-1',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 8. Evaluate the SFT Chat Model

Use a reduced 2-GPU chat evaluation first:

- source is `sft`
- load model tag `d8_kaggle`
- limit to `--max-problems=20`
- use `--batch-size=2` for categorical tasks


In [ ]:
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_eval',
    '--',
    '-i', 'sft',
    '-g', 'd8_kaggle',
    '-x', '20',
    '-b', '2',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 9. Install Distillation Extras

For teacher generation with `--load-in-8bit=1`, install `bitsandbytes` and `accelerate`.

In [ ]:
cmd = [
    sys.executable,
    '-m', 'pip',
    'install',
    '-q',
    'bitsandbytes>=0.46.1',
    'accelerate>=1.0.0',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, check=True)


## 10. Generate a Tiny Teacher SFT Dataset

Keep this first distillation-data step small:

- teacher model: `unsloth/Llama-3.2-3B-Instruct`
- input source: `gsm8k`
- output format: `sft`
- `--max-examples=16`
- `--load-in-8bit=1`


In [ ]:
cmd = [
    sys.executable,
    '-m', 'scripts.chat_distill_data',
    '--teacher-model', 'unsloth/Llama-3.2-3B-Instruct',
    '--input-source', 'gsm8k',
    '--output-format', 'sft',
    '--max-examples', '16',
    '--max-new-tokens', '128',
    '--load-in-8bit', '1',
    '--output-path', str(WORK_CACHE / 'teacher_sft.jsonl'),
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 11. Distill the SFT Student

Train a small distillation branch starting from `sft:d8_kaggle`.


In [ ]:
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_distill',
    '--',
    '--run', 'dummy',
    '--student-source', 'sft',
    '--student-tag', 'd8_kaggle',
    '--data-path', str(WORK_CACHE / 'teacher_sft.jsonl'),
    '--data-format', 'sft',
    '--num-epochs', '1',
    '--batch-size', '2',
    '--max-seq-len', '1024',
    '--eval-every', '10',
    '--save-every', '50',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 12. Generate Tiny Preference Data

Build a small preference dataset for DPO using the same teacher model.


In [ ]:
#12
cmd = [
    sys.executable,
    '-m', 'scripts.chat_distill_data',
    '--teacher-model', 'unsloth/Llama-3.2-3B-Instruct',
    '--input-source', 'gsm8k',
    '--output-format', 'preference',
    '--max-examples', '16',
    '--max-new-tokens', '128',
    '--load-in-8bit', '1',
    '--output-path', str(WORK_CACHE / 'teacher_prefs.jsonl'),
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 13. Run a Tiny DPO Branch

Start from `sft:d8_kaggle` and keep this DPO branch intentionally small.


In [ ]:
#13
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_dpo',
    '--',
    '--run', 'dummy',
    '--model-source', 'sft',
    '--model-tag', 'd8_kaggle',
    '--reference-source', 'sft',
    '--reference-tag', 'd8_kaggle',
    '--preference-source', 'jsonl',
    '--preference-path', str(WORK_CACHE / 'teacher_prefs.jsonl'),
    '--max-train-examples', '12',
    '--max-val-examples', '4',
    '--batch-size', '2',
    '--num-epochs', '1',
    '--max-steps', '8',
    '--eval-every', '4',
    '--save-every', '50',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 14. Compare SFT, Distill, and DPO

Use a small comparison run to see the three branches side by side.


In [ ]:
#14
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_post_eval',
    '--',
    '--models', f'sft=sft:d8_kaggle',
    '--models', f'distill=@{WORK_CACHE / "chatdistill_checkpoints"}:d8_kaggle',
    '--models', f'dpo=@{WORK_CACHE / "chatdpo_checkpoints"}:d8_kaggle',
    '--max-problems', '20',
    '--batch-size', '2',
]

env = os.environ.copy()
print('Running:', ' '.join(str(x) for x in cmd))
subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)


## 14.1 Inspect Saved Comparison Report

Read the report file written by `chat_post_eval.py` without rerunning the comparison.


In [ ]:
report_path = WORK_CACHE / 'report' / 'chat-post-eval.md'
print(report_path)
print(report_path.exists())

if report_path.exists():
    print(report_path.read_text())


## 15. Quantize the Distilled Model

Export a small int8 artifact from the distilled branch.


In [ ]:
#15 1 GPU
cmd = [
    sys.executable,
    '-m', 'scripts.chat_quantize',
    '--checkpoint-dir', str(WORK_CACHE / 'chatdistill_checkpoints'),
    '--model-tag', 'd8_kaggle',
    '--method', 'int8_linear',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 15.1 Tiny `chat_quant_awq.py` Code-Path Test


In [ ]:
#22 quant_awq 1 GPU
cmd = [
    sys.executable,
    '-m', 'scripts.chat_quant_awq',
    '--checkpoint-dir', str(WORK_CACHE / 'chatdistill_checkpoints'),
    '--model-tag', 'd8_kaggle',
    '--calibration-examples', '4',
    '--max-calibration-tokens', '128',
    '--suffix', 'tinytest',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 16. Evaluate the Quantized Distilled Model

Compare the distilled checkpoint against its quantized export.


In [ ]:
#16   2 GPU
quant_artifact = WORK_CACHE / 'chatquant_exports' / 'd8_kaggle_int8_linear' / 'quant_000004.pt'

cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_quant_eval',
    '--',
    '--checkpoint-dir', str(WORK_CACHE / 'chatdistill_checkpoints'),
    '--model-tag', 'd8_kaggle',
    '--quant-artifact', str(quant_artifact),
    '--max-problems', '20',
    '--batch-size', '2',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 17. Run a Tiny `chat_rl.py` Branch

This is the repo's GSM8K RL path. Keep it extremely small on Kaggle.


In [ ]:
#17 
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_rl',
    '--',
    '--run', 'dummy',
    '--model-tag', 'd8_kaggle',
    '--max-steps', '2',
    '--device-batch-size', '2',
    '--examples-per-step', '2',
    '--num-samples', '2',
    '--eval-every', '100',
    '--save-every', '100',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 18.1 Run a Tiny `chat_universal_rl.py` RLOO-KL Branch

Use a very small `rloo_kl` run as the first universal-RL test.


In [ ]:
#18.1
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_universal_rl',
    '--',
    '--run', 'dummy',
    '--model-tag', 'd8_kaggle',
    '--objective', 'rloo_kl',
    '--kl-beta', '0.02',
    '--max-steps', '2',
    '--device-batch-size', '2',
    '--examples-per-step', '2',
    '--num-samples', '2',
    '--eval-every', '100',
    '--save-every', '100',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 18.2 Run a Tiny `chat_universal_rl.py` GSPO Branch

Use a very small `gspo` run as the first GSPO code-path test.


In [ ]:
#18.2
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_universal_rl',
    '--',
    '--run', 'dummy',
    '--model-tag', 'd8_kaggle',
    '--objective', 'gspo',
    '--update-epochs', '1',
    '--clip-epsilon', '0.2',
    '--max-steps', '2',
    '--device-batch-size', '2',
    '--examples-per-step', '2',
    '--num-samples', '2',
    '--eval-every', '100',
    '--save-every', '100',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

## 19. Run a Tiny `chat_ppo.py` Branch

Use the tiny preference dataset to test PPO with reward-model training.


In [ ]:
#19
cmd = [
    'torchrun',
    '--standalone',
    '--nproc_per_node=2',
    '-m', 'scripts.chat_ppo',
    '--',
    '--run', 'dummy',
    '--policy-tag', 'd8_kaggle',
    '--reference-tag', 'd8_kaggle',
    '--preference-source', 'jsonl',
    '--preference-path', str(WORK_CACHE / 'teacher_prefs.jsonl'),
    '--max-train-examples', '12',
    '--max-val-examples', '4',
    '--rm-batch-size', '2',
    '--rm-epochs', '1',
    '--ppo-steps', '2',
    '--ppo-epochs', '1',
    '--prompt-batch-size', '2',
    '--device-batch-size', '2',
    '--eval-every', '100',
    '--save-every', '100',
]

env = os.environ.copy()
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## 20. Serve a Full-Precision Chat Model with `chat_web.py`

This launches a blocking FastAPI server. It is best used as a final manual step, not in the middle of the notebook pipeline.


In [ ]:
#20
cmd = [
    sys.executable,
    '-m', 'scripts.chat_web',
    '--checkpoint-dir', str(WORK_CACHE / 'chatdistill_checkpoints'),
    '--model-tag', 'd8_kaggle',
    '--num-gpus', '1',
    '--port', '8000',
]

print('This command starts a blocking web server:')
print(' '.join(cmd))
# subprocess.run(cmd, cwd=WORK_REPO, check=True)

## 21. Serve a Quantized Model with `chat_web_quant.py`

This also launches a blocking FastAPI server. Run it manually when you want to test the quantized artifact.


In [ ]:
#21
quant_artifact = WORK_CACHE / 'chatquant_exports' / 'd8_kaggle_int8_linear' / 'quant_000004.pt'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web_quant',
    '--quant-artifact', str(quant_artifact),
    '--num-gpus', '1',
    '--port', '8001',
]

print('This command starts a blocking quantized web server:')
print(' '.join(cmd))
# subprocess.run(cmd, cwd=WORK_REPO, check=True)